# Joining OpenAQ and weather

## Setup

In [1]:
import os
os.chdir("/home/jovyan/work")

In [2]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("join_data_notebook")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("driver.memory:", spark.sparkContext.getConf().get("spark.driver.memory", "NOT SET"))
spark

driver.memory: 6g


## Step 1: load both cleaned tables

In [3]:
openaq = spark.read.parquet("data/processed/openaq_clean")
openaq_count = openaq.count()
print("OpenAQ rows:", openaq_count)
openaq.printSchema()
openaq.show(5)

OpenAQ rows: 4075439
root
 |-- location_id: integer (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- no2: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm25: double (nullable = true)

+-----------+-------------------+-----+----+-----+----+
|location_id|       datetime_utc|  no2|  o3| pm10|pm25|
+-----------+-------------------+-----+----+-----+----+
|       2681|2018-01-06 01:00:00| 56.5|NULL| 28.3|NULL|
|       2681|2018-01-12 13:00:00| 72.0|NULL| 45.0|NULL|
|       2681|2018-01-14 23:00:00| 33.5|NULL|26.35|NULL|
|       2681|2018-01-22 00:00:00| 25.5|NULL| 1.25|NULL|
|       2681|2018-02-01 15:00:00|68.75|NULL|19.48|NULL|
+-----------+-------------------+-----+----+-----+----+
only showing top 5 rows



In [4]:
weather = spark.read.parquet("data/processed/weather_clean")
weather_count = weather.count()
print("Weather rows:", weather_count)
weather.printSchema()
weather.show(5)

Weather rows: 10384200
root
 |-- location_id: integer (nullable = true)
 |-- station: string (nullable = true)
 |-- station_latitude: double (nullable = true)
 |-- station_longitude: double (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)

+-----------+--------------------+------------------+------------------+-------------------+--------------+--------------------+-------------+--------------+
|location_id|             station|  station_latitude| station_longitude|       datetime_utc|temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|
+-----------+--------------------+------------------+------------------+-------------------+--------------+--------------------+-------------+--------------+
|       2673|Evreux  Boulevard...|49.020202317088426|1.1387134373027465|2018-01-0

## Step 2: check the join keys are actually safe to join on

In [5]:
openaq_dupe_keys = openaq.groupBy("location_id", "datetime_utc").count().filter(F.col("count") > 1).count()
weather_dupe_keys = weather.groupBy("location_id", "datetime_utc").count().filter(F.col("count") > 1).count()
print("Duplicate join keys in openaq:", openaq_dupe_keys)
print("Duplicate join keys in weather:", weather_dupe_keys)

print("openaq datetime_utc type:", dict(openaq.dtypes)["datetime_utc"])
print("weather datetime_utc type:", dict(weather.dtypes)["datetime_utc"])

Duplicate join keys in openaq: 0
Duplicate join keys in weather: 0
openaq datetime_utc type: timestamp
weather datetime_utc type: timestamp


## Step 3: check station overlap

In [6]:
openaq_stations = set(r["location_id"] for r in openaq.select("location_id").distinct().collect())
weather_stations = set(r["location_id"] for r in weather.select("location_id").distinct().collect())

print("OpenAQ stations:", len(openaq_stations))
print("Weather stations:", len(weather_stations))
print("OpenAQ stations missing from weather:", sorted(openaq_stations - weather_stations))

OpenAQ stations: 105
Weather stations: 135
OpenAQ stations missing from weather: []


## Step 4: Inner join on `(location_id, datetime_utc)`

In [7]:
joined = openaq.join(weather, on=["location_id", "datetime_utc"], how="inner")
joined_count = joined.count()
print("OpenAQ rows before join:", openaq_count)
print("Rows after join:", joined_count)
print("Dropped for no weather match:", openaq_count - joined_count, f"({(openaq_count - joined_count) / openaq_count:.1%})")
joined.printSchema()
joined.show(5)

OpenAQ rows before join: 4075439
Rows after join: 4072187
Dropped for no weather match: 3252 (0.1%)
root
 |-- location_id: integer (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- no2: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- station: string (nullable = true)
 |-- station_latitude: double (nullable = true)
 |-- station_longitude: double (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)

+-----------+-------------------+-----+----+-----+----+----------------+----------------+-----------------+--------------+--------------------+-------------+--------------+
|location_id|       datetime_utc|  no2|  o3| pm10|pm25|         station|station_latitude|station_longitude|temperature_2m|relative_humidity_2m|precipitation|wind_sp

## Step 5: sanity checks on the result

In [ ]:

print("Joined count <= OpenAQ count?", joined_count <= openaq_count)

distinct_keys = joined.select("location_id", "datetime_utc").distinct().count()
print("Joined rows:", joined_count, "| Distinct (location_id, datetime_utc) pairs:", distinct_keys)
print("Still one row per station-hour?", joined_count == distinct_keys)

Joined count <= OpenAQ count? True
Joined rows: 4072187 | Distinct (location_id, datetime_utc) pairs: 4072187
Still one row per station-hour? True


In [ ]:

joined.agg(
    F.sum(F.col("temperature_2m").isNull().cast("int")).alias("null_temperature_2m"),
    F.sum(F.col("relative_humidity_2m").isNull().cast("int")).alias("null_relative_humidity_2m"),
    F.sum(F.col("precipitation").isNull().cast("int")).alias("null_precipitation"),
    F.sum(F.col("wind_speed_10m").isNull().cast("int")).alias("null_wind_speed_10m"),
).show()

+-------------------+-------------------------+------------------+-------------------+
|null_temperature_2m|null_relative_humidity_2m|null_precipitation|null_wind_speed_10m|
+-------------------+-------------------------+------------------+-------------------+
|                  0|                        0|                 0|                  0|
+-------------------+-------------------------+------------------+-------------------+



## Step 7: save the joined table
Only run this once you're happy with the result above.

In [12]:
OUT_PATH = "data/processed/openaq_weather_joined"
joined.coalesce(8).write.mode("overwrite").parquet(OUT_PATH)
print(f"Saved to {OUT_PATH}")
print("Summary: OpenAQ had", openaq_count, "rows, joined table has", joined_count, "rows")

Saved to data/processed/openaq_weather_joined
Summary: OpenAQ had 4075439 rows, joined table has 4072187 rows
